In [4]:
import pandas as pd
import numpy as np
import os

print("Imports successful")

Imports successful


In [5]:
# Load the original cleaned dataset
original = pd.read_csv('../data/processed/cleaned_dataset.csv')

print(f"Original dataset shape: {original.shape}")
print(f"Columns: {original.columns.tolist()[:5]} ...")

# The URL column in PhiUSIIL is called 'URL' (uppercase)
# The label column uses 1=legitimate, 0=phishing (PhiUSIIL convention)
print(f"\nOriginal label distribution (PhiUSIIL convention):")
print(original['label'].value_counts())
print("In PhiUSIIL: 1=legitimate, 0=phishing")

# Extract just URL and label, rename URL to lowercase url
original_slim = original[['URL', 'label']].copy()
original_slim = original_slim.rename(columns={'URL': 'url'})

# FLIP labels to our convention: 1=phishing, 0=legitimate
original_slim['label'] = original_slim['label'].apply(lambda x: 0 if x == 1 else 1)

print(f"\nAfter flipping to our convention (1=phishing, 0=legitimate):")
print(original_slim['label'].value_counts())
print("Label 1 (phishing) should be ~100,945")
print("Label 0 (legitimate) should be ~134,850")

# Verify with sample URLs
print(f"\nSample label=1 (should look like phishing):")
for url in original_slim[original_slim['label']==1]['url'].head(3).tolist():
    print(f"  {url}")

print(f"\nSample label=0 (should look like legitimate):")
for url in original_slim[original_slim['label']==0]['url'].head(3).tolist():
    print(f"  {url}")

Original dataset shape: (235795, 56)
Columns: ['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength'] ...

Original label distribution (PhiUSIIL convention):
label
1    134850
0    100945
Name: count, dtype: int64
In PhiUSIIL: 1=legitimate, 0=phishing

After flipping to our convention (1=phishing, 0=legitimate):
label
0    134850
1    100945
Name: count, dtype: int64
Label 1 (phishing) should be ~100,945
Label 0 (legitimate) should be ~134,850

Sample label=1 (should look like phishing):
  http://www.teramill.com
  http://www.f0519141.xsph.ru
  http://www.shprakserf.gq

Sample label=0 (should look like legitimate):
  https://www.southbankmosaics.com
  https://www.uni-mainz.de
  https://www.voicefmradio.co.uk


In [6]:
phishtank = pd.read_csv('../data/raw/phishtank_verified.csv')
print(f"PhishTank columns: {phishtank.columns.tolist()}")
print(f"PhishTank rows: {len(phishtank)}")

# Find the URL column - check your column names
url_col = 'url' if 'url' in phishtank.columns else phishtank.columns[0]
print(f"Using URL column: {url_col}")

# All PhishTank URLs are phishing = label 1
phishtank_urls = pd.DataFrame({
    'url': phishtank[url_col].astype(str),
    'label': 1
})

phishtank_urls = phishtank_urls.dropna(subset=['url'])
phishtank_urls = phishtank_urls[phishtank_urls['url'].str.strip() != '']
phishtank_urls = phishtank_urls[phishtank_urls['url'] != 'nan']

print(f"\nPhishTank phishing URLs: {len(phishtank_urls)}")
print("All labelled 1 (phishing)")
print("Sample:")
for url in phishtank_urls['url'].head(3).tolist():
    print(f"  {url}")

PhishTank columns: ['phish_id', 'url', 'phish_detail_url', 'submission_time', 'verified', 'verification_time', 'online', 'target']
PhishTank rows: 64067
Using URL column: url

PhishTank phishing URLs: 64067
All labelled 1 (phishing)
Sample:
  http://allegrolokalnie.pl-214154.cfd/oferta/rower-wodny-pelikan-22/68838/dostawa
  https://gravatar.com/technicallysecret9fb064a929
  https://optusnet-rust.vercel.app


In [7]:
# Load Tranco top domains
with open('../data/raw/tranco_top_1million.txt') as f:
    tranco_domains = f.read().splitlines()

print(f"Tranco domains loaded: {len(tranco_domains)}")

# Use top 50,000 legitimate domains
tranco_sample = tranco_domains[:50000]

# Convert to full HTTPS URLs - all legitimate = label 0
tranco_urls = pd.DataFrame({
    'url': ['https://' + d for d in tranco_sample],
    'label': 0
})

print(f"Tranco legitimate URLs: {len(tranco_urls)}")
print("All labelled 0 (legitimate)")
print("Sample:")
for url in tranco_urls['url'].head(3).tolist():
    print(f"  {url}")

Tranco domains loaded: 1000000
Tranco legitimate URLs: 50000
All labelled 0 (legitimate)
Sample:
  https://google.com
  https://gtld-servers.net
  https://cloudflare.com


In [8]:
# All three sources now use same convention: 1=phishing, 0=legitimate
combined = pd.concat([
    original_slim,
    phishtank_urls,
    tranco_urls
], ignore_index=True)

print(f"Combined before dedup: {len(combined)}")
combined = combined.drop_duplicates(subset=['url'])
combined = combined.dropna()
print(f"Combined after dedup: {len(combined)}")

print(f"\nFinal label distribution (1=phishing, 0=legitimate):")
print(combined['label'].value_counts())

# Final sanity check
print(f"\nSample label=1 (should be phishing):")
for url in combined[combined['label']==1].sample(3, random_state=42)['url'].tolist():
    print(f"  {url}")

print(f"\nSample label=0 (should be legitimate):")
for url in combined[combined['label']==0].sample(3, random_state=42)['url'].tolist():
    print(f"  {url}")

Combined before dedup: 349862
Combined after dedup: 347296

Final label distribution (1=phishing, 0=legitimate):
label
0    184844
1    162452
Name: count, dtype: int64

Sample label=1 (should be phishing):
  https://oracle-ayano-9-trends.ghost.io/rubio-podium-defer-repositioning-watch-pattern/
  http://www.ssl-vaeit.com
  http://www.tomsburs.shop.co

Sample label=0 (should be legitimate):
  https://www.netlimiter.com
  https://broadstreetads.com
  https://www.fourthsource.com


In [9]:
output_path = '../data/processed/combined_dataset.csv'
combined.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Total rows: {len(combined)}")

Saved to: ../data/processed/combined_dataset.csv
Total rows: 347296
